# Replicating direct weight writing with small neural networks

A neural network makes decisions through calculations whose numerical coefficients are called weights. Usually a training program chooses those numbers by repeatedly adjusting mistakes on examples. Another possibility is to ask a language model to choose the actual coefficients directly. A useful outcome would be a complete set of numbers that makes the network perform the requested task on inputs it has not been shown.

This notebook is part of our attempt to reproduce that idea on much smaller models, motivated by [Sparks of Weight-Space Programming](https://rikiyat.github.io/blog/weight-space-programming/) (Takehi et al., 2026). The distinction is between writing the numbers and merely writing code that trains a network. Throughout these experiments, the language model authors the candidate parameters and an independent harness saves and tests them unchanged.

Start with an explicitly illustrative task: two numbers belong to class one when their signs differ, and class zero when their signs match. A human can describe that rule in one sentence. A language model that writes weights must go further: its numbers have to make the network's arithmetic distinguish the two cases. An answer that explains the rule well but supplies unusable coefficients has not completed the task.

One complication is a curved decision boundary. Points can be assigned to rings according to distance from the origin, but a network built from straight-line calculations has to approximate the curves. Another complication is learning the rule from examples rather than receiving it explicitly. These changes help distinguish a compact numerical implementation of a known rule from choosing a useful classifier from a labelled sample.

Our experiments therefore move from seventeen-value rules to a 388-value ring classifier, then to a 6,570-value handwritten-digit classifier. The digit input is an eight-by-eight grid of ink counts, and the answer is one of ten labels. For an illustrative zero, a loop of dark blocks surrounds a blank centre. A six may also contain a loop, so one pixel detector will not settle every case. The network needs combinations of evidence that remain useful when handwriting changes.

The architecture specifies which calculations happen. Each layer multiplies inputs by a weight matrix, adds biases and passes its results onward. The final layer produces scores, and the largest score identifies the predicted class. The teacher receives this exact arithmetic and must supply every coefficient. It cannot execute code, inspect files, run an optimizer or query the evaluator while writing its answer.

The linked [SQL experiment](../42-tiny-sql-direct-weights/) tests another branch of the same replication attempt: directly writing a tiny sequence generator. Together the notebooks record what worked, what failed at the task, and what stopped before any executable network existed. Preparatory pretrained-model inspection and random initialization are kept separate from evidence that a language model successfully wrote weights.

The central question is how far the direct-writing result extends when the network is smaller and the task changes. Success on one rule does not guarantee success on a sequence task, and a long numerical response can fail before its function can be tested. The public digit dataset also may be familiar from pretraining. Its score establishes what the submitted network does on these images, while leaving open how much the teacher learned from the supplied sample.

## Study summary

This component of the replication tests directly written MLPs at three sizes: seventeen parameters for explicit mathematical rules, 388 for four rings, and **6,570 for digits**. Astra succeeds on the small rule tests; the larger six-model comparison supplies the same **2,048 labelled images** and measures performance on **1,797 official test images**. Another 1,775 training images are held back as a secondary check. An MLP, or multilayer perceptron, is a network of successive fully connected layers; each neuron in a layer receives every output of the previous layer.

Accuracy counts the images whose predicted digit equals the supplied label. For example, 90 correct predictions among 100 images would be 90% accuracy. Larger values are better. A one-nearest-neighbour reference, which simply copies the label of the most similar visible image, scores 1,750/1,797, or 97.38%, on the official test split. It is a separate reference method, not a source of candidate network weights.

3 of six first submissions produced scored networks; 3 ended without a usable network. Astra classified 1645/1,797 test images correctly (91.54%). Sol classified 1422/1,797 test images correctly (79.13%). Muse Wonder classified 606/1,797 test images correctly (33.72%). Transport recoveries are reported separately below and never replace the first-attempt failure counts.

With the owner's numbered-output intervention, Gemini also produces a complete network: 4,202 indexed entries followed by 2,368 missing entries on a continuation, scoring 1,091/1,797 (60.71%). Those two indexed calls follow three earlier unusable Gemini requests. This reproduces useful direct writing under a multi-call format workflow, not a first-reply success.

This is one submission per model on an old public benchmark. The result is conditional on the architecture, the chosen sample, the prompt and each provider's output settings. Possible pretraining exposure makes this an upper bound on learning from previously unseen data. A large number of tested images does not turn six generated networks into a reliable general model ranking.

## Contents

- [1. Replication scope and data](#1-replication-scope-and-data)
- [2. The fixed network and numerical contract](#2-the-fixed-network-and-numerical-contract)
- [3. Model access and isolation](#3-model-access-and-isolation)
- [4. How the study got here](#4-how-the-study-got-here)
- [5. Results and comparisons](#5-results-and-comparisons)
- [6. Worked test cases and error analysis](#6-worked-test-cases-and-error-analysis)
- [7. Reasoning logs and parameter patterns](#7-reasoning-logs-and-parameter-patterns)
- [8. Limits and practical meaning](#8-limits-and-practical-meaning)
- [9. Reproduction, caches and spend](#9-reproduction-caches-and-spend)
- [10. Future work and further reading](#10-future-work-and-further-reading)

```mermaid
flowchart LR
    U["UCI optical digits<br/>5,620 labelled 8×8 images"] --> S["Freeze official split and sample<br/>2,048 visible; 3,572 hidden"]
    S --> P["Identical text packet<br/>visible CSV + exact inference code"]
    P --> M["6 initial calls + explicit format recoveries<br/>request 6,570 numbers"]
    M --> V["Strict parsing and unchanged loading<br/>readable network or separate failure"]
    V --> E["Fixed float64 forward pass<br/>1,797 test + 1,775 unused training cases"]
    E --> R["Accuracy, confusion, examples<br/>time, cost and emitted reasoning"]
```

One image from the visible training sample is serialized as sixty-four ink counts and a digit label. The same packet goes independently to every teacher model. A returned parameter vector is reshaped into six tensors and applied to hidden images. Their labels enter only when predictions are scored. No output from one teacher is passed to another, and the tester gives no corrective feedback during this study.

### Replay setup

The next cell defines this notebook's helpers in place. It does not import experiment code from another folder and does not invoke a teacher. Ordinary execution uses only committed data.

In [1]:
"""Direct numeric weight submissions only. No training or tester-authored candidate weights."""
from __future__ import annotations
import argparse
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime as dt
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import time
import zipfile
import numpy as np
import requests

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA = ROOT / 'data'
MODELS = {
    'astra': {'backend':'codex', 'model':'gpt-6-astra', 'effort':'xhigh'},
    'sol': {'backend':'codex', 'model':'gpt-6.1-sol', 'effort':'xhigh'},
    'opus': {'backend':'claude', 'model':'claude-opus-5-5', 'effort':'xhigh'},
    'muse': {'backend':'muse', 'model':'muse-wonder-full-1.0', 'effort':'xhigh'},
    'nemotron': {'backend':'openrouter', 'model':'nvidia/nemotron-3.5-super-vl:free', 'canonical':'nvidia/nemotron-3.5-super-vl-20261015'},
    'gemini': {'backend':'batch', 'model':'google/gemini-4-argon:batch', 'canonical':'google/gemini-4-argon-20261013', 'effort':'high'},
}
N_VISIBLE = 2048
MAX_NEW_CALLS = 1
MAX_OUTPUT_TOKENS = 131072
CLAUDE_MAX_BUDGET_USD = 5.0
SEED = 20261103
SHAPES = {'W1':[64,64], 'b1':[64], 'W2':[32,64], 'b2':[32], 'W3':[10,32], 'b3':[10]}
N_PARAMS = sum(int(np.prod(s)) for s in SHAPES.values())
assert N_PARAMS == 6570
SCHEMA = {'type':'object', 'additionalProperties':False, 'required':['weights'], 'properties':{
    'weights':{'type':'array','minItems':N_PARAMS,'maxItems':N_PARAMS,'items':{'type':'number'}}}}
PREPOST = '''import numpy as np
def preprocess(raw_pixels):
    # Exactly 64 integers, row-major flattening of the 8 by 8 image.
    # 0 is blank, 16 is the largest ink count. No fitted transform.
    return np.asarray(raw_pixels, dtype=np.float64) / 16.0
def network(raw_pixels, W1, b1, W2, b2, W3, b3):
    x = preprocess(raw_pixels)
    h1 = np.maximum(x @ W1.T + b1, 0.0)
    h2 = np.maximum(h1 @ W2.T + b2, 0.0)
    return h2 @ W3.T + b3
def postprocess(scores):
    # Digit labels are 0 through 9. Ties choose the lowest index.
    return np.argmax(scores, axis=-1)
'''

def dump(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

def sha(value):
    return hashlib.sha256(value if isinstance(value,bytes) else value.encode()).hexdigest()

def utc():
    return dt.datetime.now(dt.timezone.utc).isoformat()

def trace_call(key, config, request, record):
    """One trace per fresh call, silently optional, never called for cache replay."""
    host = os.environ.get('LANGFUSE_HOST','http://localhost:3300').rstrip('/')
    auth = (os.environ.get('LANGFUSE_PUBLIC_KEY',''),os.environ.get('LANGFUSE_SECRET_KEY',''))
    try:
        healthy = requests.get(host+'/api/public/health',timeout=2).ok
    except requests.RequestException:
        healthy = False
    if not healthy or not all(auth): return
    def attr(k,v): return {'key':k,'value':{'stringValue':v if isinstance(v,str) else json.dumps(v,default=str)}}
    notebook = '41-direct-weight-mlps'
    attrs = [attr('langfuse.observation.type','generation'),attr('langfuse.observation.model.name',config['model']),
             attr('langfuse.observation.input',request),attr('langfuse.observation.output',record.get('answer','')),
             attr('langfuse.trace.name','41 direct MLP weights'),attr('langfuse.environment','notebook'),
             attr('langfuse.trace.metadata.notebook',notebook),attr('langfuse.trace.metadata.cache_key',key),
             attr('langfuse.observation.metadata.native_usage',record.get('usage'))]
    if record.get('cost_usd') is not None: attrs.append(attr('langfuse.observation.cost_details',{'total':record['cost_usd']}))
    start = dt.datetime.fromisoformat(record['started_utc']); end = dt.datetime.fromisoformat(record['finished_utc'])
    span = {'traceId':sha(notebook+'|'+key)[:32], 'spanId':sha(notebook+'|'+key+'|gen|0')[:16],
            'name':'Write 6570 MLP parameters','kind':1,'attributes':attrs,
            'startTimeUnixNano':str(int(start.timestamp()*1e9)),'endTimeUnixNano':str(int(end.timestamp()*1e9))}
    try:
        requests.post(host+'/api/public/otel/v1/traces',json={'resourceSpans':[{'resource':{'attributes':[
            attr('service.name','experiment-notebooks')]},'scopeSpans':[{'scope':{'name':notebook},'spans':[span]}]}]},auth=auth,timeout=10)
    except requests.RequestException: pass

def prepare():
    DATA.mkdir(parents=True,exist_ok=True)
    source = DATA/'source'; source.mkdir(exist_ok=True)
    url = 'https://archive.ics.uci.edu/static/public/80/optical+recognition+of+handwritten+digits.zip'
    archive = source/'optdigits.zip'
    if not archive.exists():
        response = requests.get(url,timeout=45); response.raise_for_status(); archive.write_bytes(response.content)
    with zipfile.ZipFile(archive) as z:
        for name in ['optdigits.tra','optdigits.tes','optdigits.names']:
            (source/name).write_bytes(z.read(name))
    train = np.loadtxt(source/'optdigits.tra',delimiter=',',dtype=np.int64)
    test = np.loadtxt(source/'optdigits.tes',delimiter=',',dtype=np.int64)
    assert train.shape == (3823,65) and test.shape == (1797,65)
    rng=np.random.default_rng(SEED); selected=[]
    for label in range(10):
        selected.extend(rng.permutation(np.flatnonzero(train[:,-1]==label))[:N_VISIBLE//10+(label<N_VISIBLE%10)])
    selected=np.asarray(selected);selected=selected[rng.permutation(len(selected))]
    remaining=np.setdiff1d(np.arange(len(train)),selected)
    np.savez_compressed(DATA/'dataset.npz',visible=train[selected],visible_indices=selected,
                        hidden_train=train[remaining],hidden_train_indices=remaining,test=test)
    sample='\n'.join(','.join(map(str,row)) for row in train[selected])+'\n'
    (DATA/'visible_sample.csv').write_text(sample,encoding='utf-8')
    (DATA/'pre_post_processing.py.txt').write_text(PREPOST,encoding='utf-8')
    prompt='''You are the sole numerical weight author for a neural-network classifier.
You have NO tools, code execution, calculator, shell, file access, web, skills, plugins,
MCP or other agents. Reason internally. The tester loads your numerical output unchanged.
It will not train, optimize, correct, fill, truncate or choose any candidate parameter.

TASK: Recognize handwritten digits 0 through 9 from 8 by 8 grayscale ink-count images.
Each CSV line below has 64 pixel values in row-major order, then its integer digit label.
Pixels are integers 0..16, where 0 is blank. The 2048 lines are the ONLY labelled sample
you receive. They were selected, approximately balanced by digit, from the UCI Optical
Recognition of Handwritten Digits official training split. You will be evaluated on its
withheld 1797-image official test split and the 1775 training images not shown here.
Do not output predictions, code or a trained-model download. Your complete numbers must
implement a useful classifier in the fixed MLP below, with no external lookup at inference.

ARCHITECTURE: 64 -> 64 ReLU -> 32 ReLU -> 10 linear scores, float64 computation.
There are exactly 6570 independent editable weights and biases. No other inference
operations are permitted besides the exact preprocessing and postprocessing below.
Sparse constructions and zero biases are allowed; there is no density or Gaussianity
requirement. Prefer numerically moderate coefficients, but every finite value is allowed.

EXACT PREPROCESSING, FORWARD COMPUTATION AND POSTPROCESSING (read, do not execute):
'''+PREPOST+'''
RETURN ONLY {"weights":[6570 explicit finite JSON numbers]}.
Flatten tensors in the following order; flatten each tensor in C/row-major order.
W1 [64,64]: offsets 0..4095, then b1 [64]: 4096..4159;
W2 [32,64]: 4160..6207, then b2 [32]: 6208..6239;
W3 [10,32]: 6240..6559, then b3 [10]: 6560..6569.
Rows of a weight matrix are output neurons; columns are input neurons.
Return ALL 6570 numbers, including zeros. No prose, Markdown, code, formulas, ellipses,
repetition notation, omitted entries, extra keys, or special NaN/Infinity values.
There is one submission and no intermediate feedback. Maximize held-out accuracy.

LABELLED SAMPLE CSV (columns p00,p01,...,p63,label):
'''+sample+'\nEND SAMPLE. Return only the complete JSON weights object.\n'
    (DATA/'prompt.txt').write_text(prompt,encoding='utf-8');dump(DATA/'weights_schema.json',SCHEMA)
    dataset_keys={tuple(row[:64]) for row in train[selected]}
    manifest={'protocol':'digits-v1','prepared_utc':utc(),'source_url':url,
        'source_page':'https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits',
        'creators':'E. Alpaydin and C. Kaynak','source_donation_date':'1998-06-30','licence':'CC BY 4.0',
        'freshness_exception':'User explicitly requested public data. Old public benchmark may have been in teacher training; treat scores as upper bounds.',
        'archive_sha256':sha(archive.read_bytes()),'file_sha256':{name:sha((source/name).read_bytes()) for name in ['optdigits.tra','optdigits.tes','optdigits.names']},
        'architecture':[64,64,32,10],'shapes':SHAPES,'parameters':N_PARAMS,'seed':SEED,
        'n_visible':len(selected),'n_hidden_train':len(remaining),'n_official_test':len(test),
        'visible_class_counts':np.bincount(train[selected,-1],minlength=10).tolist(),
        'test_class_counts':np.bincount(test[:,-1],minlength=10).tolist(),
        'test_pixel_rows_duplicated_in_visible':sum(tuple(row[:64]) in dataset_keys for row in test),
        'prompt_sha256':sha(prompt),'prompt_bytes':len(prompt.encode()),'models':MODELS,
        'no_tools':True,'training_steps':0,'tester_authors_weights':False,'one_submission_per_model':True,
        'primary_metric':'official test accuracy; Wilson 95% interval conditional on this single submitted network',
        'secondary':['visible-sample accuracy','unused-training accuracy','per-class accuracy','confusion matrix','weight statistics'],
        'no_content_retries':True,'no_cancellation_on_task_switch':True,
        'reasoning_policy':'Save raw provider-emitted reasoning, deltas and opaque details; missing reasoning is not reconstructed.',
        'user_authorized_closed_openrouter_exception':'google/gemini-4-argon:batch explicitly requested for this experiment only.'}
    path=DATA/'manifest.json'
    if path.exists(): assert json.loads(path.read_text(encoding='utf-8'))['prompt_sha256']==manifest['prompt_sha256']
    else: dump(path,manifest)
    print(json.dumps({k:manifest[k] for k in ['parameters','n_visible','n_hidden_train','n_official_test','prompt_bytes','test_pixel_rows_duplicated_in_visible']}),flush=True)

def private_dir(key):
    path=Path(tempfile.gettempdir())/'direct-weights-41'/key
    (path/'empty').mkdir(parents=True,exist_ok=True)
    (path/'neutral.txt').write_text('You are a numerical weight author. Return only the requested JSON data.\n',encoding='utf-8')
    dump(path/'empty-mcp.json',{'mcpServers':{}})
    return path

def cli_command(key, out):
    cfg=MODELS[key];private=private_dir(key)
    if cfg['backend']=='codex':
        cmd=json.loads((DATA/'transport'/'codex_template.json').read_text(encoding='utf-8'))
        if '--ephemeral' in cmd:cmd.remove('--ephemeral')
        cmd[cmd.index('-m')+1]=cfg['model'];cmd[cmd.index('-C')+1]=str(private/'empty')
        cmd[cmd.index('--output-schema')+1]=str(DATA/'weights_schema.json')
        for i,arg in enumerate(cmd):
            if arg.startswith('model_reasoning_effort='):cmd[i]='model_reasoning_effort='+json.dumps(cfg['effort'])
            if arg.startswith('model_instructions_file='):cmd[i]='model_instructions_file='+json.dumps(str(private/'neutral.txt'))
        return cmd[:-1]+['-c','model_context_window=1050000','-c','model_auto_compact_token_limit=1000000',
                        '--output-last-message',str(out/'cli_last_message.json'),'-']
    if cfg['backend']=='claude':
        return [shutil.which('claude'),'--print','--model',cfg['model'],'--effort',cfg['effort'],'--tools','',
                '--safe-mode','--disable-slash-commands','--strict-mcp-config','--mcp-config',str(private/'empty-mcp.json'),
                '--disallowedTools','mcp__*','--system-prompt',(private/'neutral.txt').read_text(),
                '--output-format','stream-json','--verbose','--include-partial-messages','--max-budget-usd',str(CLAUDE_MAX_BUDGET_USD)]
    exe=os.environ.get('MUSE_EXECUTABLE') or shutil.which('muse.cmd') or shutil.which('muse')
    return [exe,'exec','--preset','empty','--disable-web-tools','--disable-reminders','--disable-request-user-input',
            '--no-foreign-personal-context','--disable-shell','--disable-write','--model',cfg['model'],
            '--reasoning-effort',cfg['effort'],'--max-model-steps','1','--workspace',str(private/'empty'),
            '--output-schema',str(DATA/'weights_schema.json'),'--prompt-file',str(DATA/'prompt.txt'),'--json']

def parse_cli(backend, events):
    answer='';reasoning=[];usage={};completed=False;tools=[];cost=None;resolved=[];catalogues=[]
    if backend=='codex':
        for e in events:
            if e.get('type')=='item.completed':
                item=e.get('item',{});kind=item.get('type')
                if kind=='agent_message':answer=item.get('text','')
                elif kind=='reasoning' and item.get('text'):reasoning.append(item['text'])
                elif kind not in ['reasoning','error']:tools.append(item)
            if e.get('type')=='turn.completed':usage=e.get('usage',{});completed=True
    elif backend=='claude':
        for e in events:
            if e.get('type')=='system' and e.get('subtype')=='init':
                catalogues.append({k:e.get(k) for k in ['tools','mcp_servers','skills','plugins']});resolved.append(e.get('model'))
            if e.get('type')=='assistant':
                for block in e.get('message',{}).get('content',[]):
                    if block.get('type')=='thinking':reasoning.append(block.get('thinking',''))
                    if block.get('type')=='tool_use':tools.append(block)
            if e.get('type')=='result':
                answer=e.get('result','');usage=e.get('usage',{});cost=e.get('total_cost_usd')
                completed=e.get('subtype')=='success' and not e.get('is_error',False)
    else:
        for e in events:
            kind=e.get('payload_type','');p=e.get('payload',{})
            if kind=='run.terminal.completed':answer=p.get('text','');completed=True
            if kind=='run.usage':usage=p
            if 'reasoning' in kind or 'thinking' in kind:
                reasoning.append(p.get('text') or p.get('delta') or json.dumps(p))
            if kind.startswith('tool.') or 'tool_call' in kind:tools.append(e)
    return dict(answer=answer,reasoning='\n'.join(reasoning),usage=usage,completed=completed,
                tool_events=tools,cost_usd=cost,reported_models=resolved,catalogues=catalogues)

def run_cli(key, out, request):
    cfg=MODELS[key];cmd=request['command'];private=private_dir(key)
    env=os.environ.copy()
    for name in list(env):
        if name.startswith(('CODEX_','CLAUDE_')) or name in ['OPENAI_API_KEY','ANTHROPIC_API_KEY','OPENROUTER_API_KEY']:env.pop(name,None)
    env['PYTHONIOENCODING']='utf-8'
    credential=None
    if cfg['backend']=='codex':
        flags=[]
        for i,arg in enumerate(cmd):
            if arg=='-c':flags.extend(['-c',cmd[i+1]])
        original=Path(os.environ.get('CODEX_HOME',str(Path.home()/'.codex')))
        env['CODEX_HOME']=str(private)
        pre=subprocess.run([cmd[0],'-C',str(private/'empty')]+flags+['debug','prompt-input','PREFLIGHT; NO MODEL CALL.'],
                           env=env,capture_output=True,text=True,encoding='utf-8',timeout=60)
        (out/'preflight.json').write_text(pre.stdout,encoding='utf-8')
        assert pre.returncode==0,pre.stderr
        assert not any(s in pre.stdout for s in ['<skills_instructions>','### Available skills','# AGENTS.md instructions'])
        credential=private/'auth.json';assert not credential.exists(),'Unresolved prior private credential'
        shutil.copyfile(original/'auth.json',credential)
    if cfg['backend']=='claude':env['CLAUDE_CODE_MAX_OUTPUT_TOKENS']=str(request.get('output_token_cap',64000))
    events=[];started=utc();tick=time.monotonic()
    try:
        with (out/'events.jsonl').open('w',encoding='utf-8') as log,(out/'stderr.txt').open('w',encoding='utf-8') as err:
            child=subprocess.Popen(cmd,cwd=private/'empty',env=env,stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                                   stderr=err,text=True,encoding='utf-8',bufsize=1)
            dump(out/'status.json',{'status':'running','started_utc':started,'pid':child.pid,'model':cfg['model']})
            if cfg['backend']!='muse':child.stdin.write(request['prompt'])
            child.stdin.close()
            for line in child.stdout:
                log.write(line);log.flush()
                try:events.append(json.loads(line))
                except json.JSONDecodeError:pass
            code=child.wait()
    finally:
        if credential is not None and credential.exists():credential.unlink()
    parsed=parse_cli(cfg['backend'],events)
    return dict(parsed,events=events,exit_code=code,started_utc=started,finished_utc=utc(),wall_seconds=time.monotonic()-tick)

def openrouter_body(key,prompt):
    cfg=MODELS[key]
    body={'model':cfg['model'] if key=='nemotron' else cfg['canonical'],'messages':[{'role':'user','content':prompt}],
          'temperature':0,'max_tokens':MAX_OUTPUT_TOKENS,'reasoning':{'enabled':True,'exclude':False},
          'response_format':{'type':'json_schema','json_schema':{'name':'mlp_weights','strict':True,'schema':SCHEMA}},
          'tool_choice':'none','plugins':[]}
    if key=='gemini':body['reasoning']={'effort':'high','exclude':False}
    return body

def run_openrouter(key,out,request):
    cfg=MODELS[key];headers={'Authorization':'Bearer '+os.environ['OPENROUTER_API_KEY']}
    started=utc();tick=time.monotonic();events=[];parts=[];reasons=[];usage={};tools=[];completed=False;error=None
    if cfg['backend']=='batch':
        submit=out/'batch_submission.json'
        if submit.exists():batch=json.loads(submit.read_text(encoding='utf-8'))
        else:
            response=requests.post('https://openrouter.ai/api/v1/batches',headers=headers,json=request['body'],timeout=(30,120))
            batch=response.json();dump(submit,batch)
            assert response.ok,(response.status_code,batch)
        assert batch.get('id'),batch
        dump(out/'status.json',{'status':'batch_pending','id':batch['id'],'started_utc':started})
        with (out/'batch_polls.jsonl').open('a',encoding='utf-8') as log:
            while batch.get('status') not in ['completed','failed','cancelled','expired']:
                time.sleep(30)
                response=requests.get('https://openrouter.ai/api/v1/batches/'+batch['id'],headers=headers,timeout=30)
                response.raise_for_status();batch=response.json()
                log.write(json.dumps(batch,ensure_ascii=False)+'\n');log.flush();dump(out/'batch_latest.json',batch)
        result=(batch.get('results') or [{}])[0]
        data=(result.get('response') or {}).get('body',{})
        dump(out/'batch_result.json',batch);events=[data];usage=data.get('usage',{})
        choice=(data.get('choices') or [{}])[0];message=choice.get('message',{})
        parts=[message.get('content') or ''];reasons=[message.get('reasoning') or message.get('reasoning_content') or '']
        tools=message.get('tool_calls') or [];completed=batch.get('status')=='completed' and choice.get('finish_reason')=='stop'
        error=result.get('error') or batch.get('error')
    else:
        body=request['body']
        with (out/'events.jsonl').open('w',encoding='utf-8') as log,(out/'reasoning.txt').open('w',encoding='utf-8') as reasonlog, \
             (out/'weights_output.json').open('w',encoding='utf-8') as textlog:
            with requests.post('https://openrouter.ai/api/v1/chat/completions',headers=headers,json=body,stream=True,timeout=(30,300)) as response:
                if not response.ok: error={'status':response.status_code,'body':response.text}
                else:
                    for line in response.iter_lines(chunk_size=1,decode_unicode=True):
                        if not line or not line.startswith('data:'):continue
                        if line[5:].strip()=='[DONE]':completed=True;break
                        e=json.loads(line[5:]);events.append(e);log.write(json.dumps(e,ensure_ascii=False)+'\n');log.flush()
                        if e.get('usage'):usage=e['usage']
                        if e.get('error'):error=e['error']
                        for choice in e.get('choices',[]):
                            delta=choice.get('delta',{})
                            content=delta.get('content') or '';reason=delta.get('reasoning') or delta.get('reasoning_content') or ''
                            if not reason:reason=''.join(x.get('text',x.get('summary','')) for x in delta.get('reasoning_details',[]) if x.get('type') in ['reasoning.text','reasoning.summary'])
                            parts.append(content);reasons.append(reason);textlog.write(content);textlog.flush();reasonlog.write(reason);reasonlog.flush()
                            if delta.get('tool_calls'):tools.append(delta['tool_calls'])
                            if choice.get('finish_reason') and choice['finish_reason']!='stop':error={'finish_reason':choice['finish_reason']}
    native_cost=(batch.get('usage') or {}).get('cost') if cfg['backend']=='batch' else usage.get('cost')
    return {'answer':''.join(parts),'reasoning':''.join(reasons),'usage':usage,'completed':completed and not error,
            'tool_events':tools,'events':events,'error':error,'started_utc':started,'finished_utc':utc(),
            'wall_seconds':time.monotonic()-tick,'cost_usd':native_cost}

def collect_one(key):
    cfg=MODELS[key];out=DATA/'responses'/key;out.mkdir(parents=True,exist_ok=True)
    prompt=(DATA/'prompt.txt').read_text(encoding='utf-8')
    assert sha(prompt)==json.loads((DATA/'manifest.json').read_text(encoding='utf-8'))['prompt_sha256']
    request={'config':cfg,'prompt':prompt,'schema':SCHEMA,'tools_enabled':[],'protocol':'digits-v1'}
    if cfg['backend'] in ['codex','claude','muse']:request['command']=cli_command(key,out)
    else:
        meta=requests.get('https://openrouter.ai/api/v1/models',timeout=30).json()['data']
        model=next(m for m in meta if m['id']==cfg['model']);assert model['canonical_slug']==cfg['canonical'];dump(out/'model_metadata.json',model)
        body=openrouter_body(key,prompt)
        if key=='nemotron':
            assert all(float(model['pricing'][k])==0 for k in ['prompt','completion'])
            body.update(provider={'only':['nvidia'],'allow_fallbacks':False,'max_price':{'prompt':0,'completion':0,'request':0}},
                        stream=True,stream_options={'include_usage':True})
        else:body={'endpoint':'/v1/chat/completions','model':cfg['canonical'],
                   'provider':{'only':['google-vertex'],'allow_fallbacks':False},'completion_window':'24h',
                   'requests':[{'custom_id':'digits-v1-gemini','body':body}]}
        request['body']=body
    keyhash=sha(json.dumps(request,sort_keys=True));cache=out/'response.json'
    if cache.exists():
        record=json.loads(cache.read_text(encoding='utf-8'));assert record['cache_key']==keyhash
        validate_and_score(key,record);return
    assert not (out/'started.json').exists(),'Prior request unresolved; inspect logs instead of regenerating'
    dump(out/'request.json',request);dump(out/'started.json',{'started_utc':utc(),'cache_key':keyhash})
    # Health check before a fresh call. Keys are never saved.
    try:requests.get(os.environ.get('LANGFUSE_HOST','http://localhost:3300').rstrip('/')+'/api/public/health',timeout=2)
    except requests.RequestException:pass
    print('START '+key,flush=True)
    try:
        record=run_cli(key,out,request) if cfg['backend'] in ['codex','claude','muse'] else run_openrouter(key,out,request)
    except Exception as exc:
        dump(out/'failure.json',{'exception':type(exc).__name__,'message':str(exc),'at_utc':utc()});raise
    record.update(cache_key=keyhash,request=request);dump(cache,record)
    (out/'weights_output.json').write_text(record.get('answer',''),encoding='utf-8')
    (out/'reasoning.txt').write_text(record.get('reasoning',''),encoding='utf-8')
    trace_call(keyhash,cfg,request,record)
    validate_and_score(key,record)
    print('FINISH '+key,flush=True)

def load_submission(text):
    payload=json.loads(text,parse_constant=lambda x: (_ for _ in ()).throw(ValueError('Non-finite JSON value')))
    assert isinstance(payload,dict) and set(payload)=={'weights'},'Wrong JSON keys'
    values=payload['weights'];assert isinstance(values,list),'Weights must be a JSON array'
    assert len(values)==N_PARAMS,f'Expected {N_PARAMS} weights; received {len(values)}'
    assert all(type(v) in [int,float] for v in values),'Non-numeric weight'
    flat=np.asarray(values,dtype=np.float64);assert np.isfinite(flat).all(),'Non-finite weight'
    arrays={};offset=0
    for name,shape in SHAPES.items():
        size=int(np.prod(shape));arrays[name]=flat[offset:offset+size].reshape(shape);offset+=size
    assert np.array_equal(np.concatenate([a.ravel() for a in arrays.values()]),flat)
    return arrays,flat

def forward(raw, arrays):
    x=np.asarray(raw,dtype=np.float64)/16.0
    h1=np.maximum(x@arrays['W1'].T+arrays['b1'],0)
    h2=np.maximum(h1@arrays['W2'].T+arrays['b2'],0)
    return h2@arrays['W3'].T+arrays['b3']

def wilson(k,n):
    z=1.959963984540054;p=k/n;den=1+z*z/n
    centre=(p+z*z/(2*n))/den;half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return [centre-half,centre+half]

def validate_and_score(key,record,out_override=None):
    out=Path(out_override) if out_override else DATA/'responses'/key
    result={'model':MODELS[key]['model'],'model_key':key,'format_failure':None,'failure_category':None,'tool_events':len(record.get('tool_events',[])),
            'tester_changed_weights':False,'training_steps':0,'parameters':N_PARAMS,
            'usage':record.get('usage'),'cost_usd':record.get('cost_usd'),'wall_seconds':record.get('wall_seconds'),
            'reasoning_characters':len(record.get('reasoning',''))}
    try:
        assert record.get('completed') and record.get('exit_code',0)==0,'Incomplete or failed model response'
        assert not record.get('tool_events'),'Tool protocol failure'
        for cat in record.get('catalogues',[]):
            assert not any(cat.get(k) for k in ['tools','mcp_servers','skills']),'Nonempty tool/customization catalogue'
            assert all(p.get('path')=='builtin' for p in cat.get('plugins',[])),'Unexpected non-builtin plugin'
        arrays,flat=load_submission(record['answer']);np.savez_compressed(out/'candidate.npz',**arrays)
        with np.load(out/'candidate.npz') as saved:assert all(np.array_equal(saved[k],v) for k,v in arrays.items())
        result['weights']={'zeros':int(np.sum(flat==0)),'max_abs':float(np.abs(flat).max()),'mean':float(flat.mean()),'rms':float(np.sqrt(np.mean(flat**2)))}
        with np.load(DATA/'dataset.npz') as data:
            predictions={}
            for split in ['visible','hidden_train','test']:
                rows=data[split];scores=forward(rows[:,:64],arrays);assert np.isfinite(scores).all(),'Non-finite network scores'
                pred=scores.argmax(axis=1);gold=rows[:,-1];correct=pred==gold
                confusion=np.zeros((10,10),dtype=int);np.add.at(confusion,(gold,pred),1)
                result[split]={'n':len(gold),'correct':int(correct.sum()),'accuracy':float(correct.mean()),
                               'wilson_95':wilson(int(correct.sum()),len(gold)),'confusion':confusion.tolist(),
                               'per_class_accuracy':(np.diag(confusion)/confusion.sum(axis=1)).tolist()}
                predictions[split+'_pred']=pred;predictions[split+'_scores']=scores
            np.savez_compressed(out/'predictions.npz',**predictions)
    except (AssertionError,ValueError,TypeError,OverflowError) as exc:
        result['format_failure']=str(exc) or type(exc).__name__
        category='unreadable_format'
        terminal=[ev for ev in record.get('events',[]) if ev.get('type')=='result']
        finish=[c.get('finish_reason') for ev in record.get('events',[]) for c in ev.get('choices',[]) if c.get('finish_reason')]
        if terminal and terminal[-1].get('safety_stops') and record.get('answer','').strip():category='refusal'
        elif terminal and terminal[-1].get('subtype')=='error_max_budget_usd':category='budget_or_output_stop'
        elif 'length' in finish or (record.get('error') or {}).get('finish_reason')=='length':category='output_limit'
        elif record.get('error'):category='provider_or_transport_failure'
        elif not record.get('completed'):category='incomplete_response'
        elif result['format_failure'].startswith('Expected 6570 weights'):category='wrong_weight_count'
        result['failure_category']=category
    dump(out/'results.json',result)
    dump(out/'status.json',{'status':'scored' if result['format_failure'] is None else 'unreadable','finished_utc':utc(),'format_failure':result['format_failure']})
    return result

# Reproduction is offline; collection helpers are defined but never called.
MAX_NEW_CALLS = 0
ALLOW_NEW_REQUESTS = False
"""Cache-only figures and measurements. No teacher calls and no candidate fitting."""
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA = ROOT/'data'
FIGURES = ROOT/'figures'
MODEL_LABELS = {'astra':'Astra','sol':'Sol','opus':'Opus 5.5','muse':'Muse Wonder','nemotron':'Nemotron','gemini':'Gemini batch'}
plt.rcParams.update({'figure.dpi':130,'savefig.dpi':160,'font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'axes.titleweight':'bold','axes.labelsize':10,
                     'figure.facecolor':'white','axes.facecolor':'white'})

def save_json(path,obj):
    path.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')

def live_reasoning_logs():
    """Only text the external provider actually emitted. Never infer hidden reasoning."""
    for key in MODEL_LABELS:
        folder=DATA/'responses'/key;events_path=folder/'events.jsonl'
        if not events_path.exists():continue
        events=[]
        for line in events_path.read_text(encoding='utf-8').splitlines():
            try:events.append(json.loads(line))
            except ValueError:pass
        if key=='opus':
            chunks=[e.get('event',{}).get('delta',{}).get('thinking','') for e in events
                    if e.get('event',{}).get('delta',{}).get('type')=='thinking_delta']
            full=[b.get('thinking','') for e in events if e.get('type')=='assistant'
                  for b in e.get('message',{}).get('content',[]) if b.get('type')=='thinking' and b.get('thinking','').strip()]
            (folder/'reasoning.txt').write_text('\n'.join(full) if full else ''.join(chunks),encoding='utf-8')
        elif key=='muse':
            chunks=[]
            for e in events:
                p=e.get('payload',{});kind=e.get('payload_type','')
                if 'reasoning' in kind or 'thinking' in kind:
                    value=p.get('text') or p.get('delta')
                    if isinstance(value,str):chunks.append(value)
            if chunks:(folder/'reasoning.txt').write_text(''.join(chunks),encoding='utf-8')

def collect_results():
    results=[]
    for key,label in MODEL_LABELS.items():
        folder=DATA/'responses'/key;path=folder/'results.json'
        result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
        result['label']=label
        result['candidate_folder']=str(folder.relative_to(DATA))
        rp=folder/'reasoning.txt';result['reasoning_characters']=len(rp.read_text(encoding='utf-8')) if rp.exists() else 0
        results.append(result)
    for recovery_key in ['opus','gemini','gemini-low']:
        key=recovery_key.split('-')[0];folder=DATA/'recovery'/recovery_key
        if not folder.exists():continue
        path=folder/'results.json'
        result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
        result['label']=MODEL_LABELS[key]+(' continuation' if key=='opus' else (' low effort' if recovery_key=='gemini-low' else ' schema retry'))
        result['candidate_folder']=str(folder.relative_to(DATA));result['recovery']=True
        rp=folder/'reasoning.txt';result['reasoning_characters']=len(rp.read_text(encoding='utf-8').strip()) if rp.exists() else 0
        results.append(result)
    indexed_root=DATA/'indexed'
    if indexed_root.exists():
        for folder in sorted(indexed_root.iterdir()):
            if not folder.is_dir() or folder.name not in MODEL_LABELS:continue
            key=folder.name;path=folder/'results.json'
            result=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'model_key':key,'format_failure':None,'pending':True}
            result['label']=MODEL_LABELS[key]+' indexed';result['indexed']=True
            result['candidate_folder']=str(folder.relative_to(DATA));result['outer_requests']=len(list(folder.glob('turn-*/response.json')))
            result['reasoning_characters']=len((folder/'reasoning.txt').read_text(encoding='utf-8').strip()) if (folder/'reasoning.txt').exists() else 0
            results.append(result)
    return results

def nearest_neighbour_baseline(dataset):
    visible=dataset['visible'];x=visible[:,:64].astype(float)/16;labels=visible[:,-1]
    metrics={};predictions={}
    for split in ['hidden_train','test']:
        z=dataset[split][:,:64].astype(float)/16;pred=[]
        for i in range(0,len(z),128):
            batch=z[i:i+128];distance=(batch**2).sum(1)[:,None]+(x**2).sum(1)[None,:]-2*batch@x.T
            pred.extend(labels[np.argmin(distance,axis=1)])
        pred=np.asarray(pred);gold=dataset[split][:,-1];predictions[split]=pred
        metrics[split]={'n':len(gold),'correct':int(np.sum(gold==pred)),'accuracy':float(np.mean(gold==pred))}
    np.savez_compressed(DATA/'nearest_neighbour_predictions.npz',**predictions)
    save_json(DATA/'nearest_neighbour_baseline.json',metrics)
    return metrics

def verify_legacy_mlps():
    results=[];legacy=DATA/'legacy'
    points=np.load(legacy/'mlp17/data/evaluation_points.npz')
    for name in ['astra','nemotron-no-reasoning']:
        folder=legacy/'mlp17/data'/name
        if not (folder/'classification.npz').exists():continue
        row={'model':name,'parameters':17}
        for task in ['classification','regression']:
            with np.load(folder/(task+'.npz')) as w:
                x=points[task+'_x'];scores=np.maximum(x@w['W1'].T+w['b1'],0)@w['W2']+w['b2'][0]
            gold=points[task+'_y'];row[task+'_n']=len(gold)
            if task=='classification':row['correct']=int(np.sum((scores>0)==gold));row['accuracy']=float(np.mean((scores>0)==gold))
            else:row['regression_mae']=float(np.mean(np.abs(scores-gold)))
        results.append(row)
    points.close()
    with np.load(legacy/'rings388/data/test_points.npz') as points:
        for name in ['astra','nemotron-free']:
            with np.load(legacy/'rings388/data'/name/'candidate.npz') as w:
                x=points['x'];h1=np.maximum(x@w['W1'].T+w['b1'],0);h2=np.maximum(h1@w['W2'].T+w['b2'],0)
                pred=(h2@w['W3'].T+w['b3']).argmax(axis=1);gold=points['y']
            expected_path=legacy/'rings388/data'/name/'results.json'
            if not expected_path.exists():expected_path=legacy/'rings388/results.json'
            expected=json.loads(expected_path.read_text())
            correct=int(np.sum(pred==gold));assert correct==expected['correct']
            results.append({'model':name,'parameters':388,'classification_n':len(gold),'correct':correct,'accuracy':float(np.mean(pred==gold))})
    save_json(DATA/'legacy_replay.json',results)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for ax,size,chance,title in zip(axes,[17,388],[50,25],['Known opposite-sign rule','Known four-ring rule']):
        subset=[r for r in results if r['parameters']==size]
        ax.bar([r['model'].replace('nemotron-no-reasoning','Nemotron\nreasoning off').replace('nemotron-free','Nemotron free').replace('astra','Astra') for r in subset],
               [100*r['accuracy'] for r in subset],color=['#286f9e','#7f9a61'])
        ax.axhline(chance,color='#777',linestyle='--');ax.set(ylim=(0,110),ylabel='Cases classified correctly (%)',title=f'{size} parameters: {title}')
        for j,r in enumerate(subset):ax.text(j,100*r['accuracy']+3,f"{r['correct']:,}/{r['classification_n']:,}",ha='center',fontsize=9)
    fig.tight_layout();fig.savefig(FIGURES/'legacy_accuracy.png',bbox_inches='tight');plt.close(fig)
    return results

def make_figures():
    FIGURES.mkdir(exist_ok=True);live_reasoning_logs();indexed_workflow_metadata();reported_billing();verify_indexed_artifacts()
    with np.load(DATA/'dataset.npz') as saved:dataset={k:saved[k] for k in saved.files}
    results=collect_results();valid=[r for r in results if not r.get('pending') and r.get('format_failure') is None]
    legacy=verify_legacy_mlps()
    baseline=nearest_neighbour_baseline(dataset)
    parameter_rows=[]
    for r in valid:
        with np.load(DATA/r['candidate_folder']/'candidate.npz') as weights:
            parameter_rows.append({'label':r['label'],'zeros':r['weights']['zeros'],'max_abs':r['weights']['max_abs'],
                'h1_nonzero_rows':int(np.sum(np.any(weights['W1']!=0,axis=1)|(weights['b1']!=0))),
                'h2_nonzero_rows':int(np.sum(np.any(weights['W2']!=0,axis=1)|(weights['b2']!=0))),
                'nonzero_by_tensor':{k:int(np.count_nonzero(weights[k])) for k in weights.files}})
    save_json(DATA/'parameter_use.json',parameter_rows)
    astra_path=DATA/'responses/astra/candidate.npz';sol_path=DATA/'responses/sol/candidate.npz'
    if astra_path.exists() and sol_path.exists():
        structure={'analysis':'Post-hoc interpretation, not teacher reasoning text','identical_downstream':{},'max_identity_error':{}}
        with np.load(astra_path) as first,np.load(sol_path) as second:
            structure['identical_downstream']={k:bool(np.array_equal(first[k],second[k])) for k in ['W2','b2','W3','b3']}
            for name,w in [('Astra',first),('Sol',second)]:
                x=dataset['test'][:,:64].astype(float)/16;h1=np.maximum(x@w['W1'].T+w['b1'],0)
                blocks=h1[:,:40].reshape(-1,10,4)
                interpreted=np.maximum(blocks[:,:,:2].sum(axis=2),blocks[:,:,2:].sum(axis=2))
                h2=np.maximum(h1@w['W2'].T+w['b2'],0);actual=h2@w['W3'].T+w['b3']
                structure['max_identity_error'][name]=float(np.max(np.abs(actual-interpreted)))
        save_json(DATA/'program_structure.json',structure)
    if parameter_rows:
        fig,ax=plt.subplots(figsize=(10,4));bottom=np.zeros(len(parameter_rows));colours=['#286f9e','#7f9a61','#b7752c']
        for names,label,colour in zip([['W1','b1'],['W2','b2'],['W3','b3']],['Input layer','Second layer','Output layer'],colours):
            counts=np.array([sum(r['nonzero_by_tensor'][k] for k in names) for r in parameter_rows]);ax.bar(range(len(counts)),counts,bottom=bottom,color=colour,label=label);bottom+=counts
        ax.set(xticks=range(len(parameter_rows)),xticklabels=[r['label'] for r in parameter_rows],ylabel='Nonzero values (of 6,570 supplied)',title='The numerical authors use different fractions of the available network')
        ax.legend(fontsize=8);fig.tight_layout();fig.savefig(FIGURES/'parameter_use.png',bbox_inches='tight');plt.close(fig)
    fig,axes=plt.subplots(2,5,figsize=(10,4.5))
    for digit,ax in enumerate(axes.flat):
        i=int(np.flatnonzero(dataset['visible'][:,-1]==digit)[0]);row=dataset['visible'][i]
        ax.imshow(row[:64].reshape(8,8),cmap='Greys',vmin=0,vmax=16,interpolation='nearest')
        ax.set_title(f'Digit {digit}: train-{int(dataset["visible_indices"][i]):04d}',fontsize=9);ax.axis('off')
    fig.suptitle('The visible sample contains ten handwritten digit classes',fontsize=13)
    fig.tight_layout();fig.savefig(FIGURES/'visible_digits.png',bbox_inches='tight');plt.close(fig)
    fig,ax=plt.subplots(figsize=(13,5));labels=[];heights=[];colours=[]
    for r in results:
        short={'Muse Wonder':'Muse\nWonder','Gemini batch':'Gemini\ninitial','Gemini batch schema retry':'Gemini\nschema retry',
               'Gemini batch low effort':'Gemini\nlow effort','Opus 5.5 continuation':'Opus 5.5\ncontinuation',
               'Gemini batch indexed':'Gemini\nindexed','Nemotron indexed':'Nemotron\nindexed'}
        labels.append(short.get(r['label'],r['label']));heights.append(100*r['test']['accuracy'] if r in valid else 0)
        colours.append('#286f9e' if r in valid else '#c2c8cf')
    labels.append('1-nearest\nneighbour');heights.append(100*baseline['test']['accuracy']);colours.append('#b7752c')
    bars=ax.bar(np.arange(len(labels)),heights,color=colours)
    for i,r in enumerate(results):
        if r in valid:
            lo,hi=np.asarray(r['test']['wilson_95'])*100;h=heights[i]
            ax.errorbar(i,h,yerr=[[h-lo],[hi-h]],fmt='none',ecolor='#172e40',capsize=3)
            ax.text(i,h+3,f'{h:.1f}%',ha='center',fontsize=9)
        else:ax.text(i,3,'Pending' if r.get('pending') else 'Unreadable',rotation=60,ha='center',fontsize=9,color='#525b65')
    ax.text(len(labels)-1,heights[-1]+3,f'{heights[-1]:.1f}%',ha='center',fontsize=9)
    ax.axhline(10,color='#777777',linestyle='--',label='Uniform guess: 10%')
    ax.set(xticks=np.arange(len(labels)),xticklabels=labels,ylim=(0,110),ylabel='Official test images classified correctly (%)')
    ax.set_title('Identical MLP architecture yields different digit-classification accuracy');ax.legend(loc='upper left',fontsize=8)
    ax.text(.01,-.25,'Grey entries have no scored network; their height is not an accuracy estimate.\nIntervals: Wilson 95%, conditional on each fixed submitted network.',transform=ax.transAxes,fontsize=8)
    fig.tight_layout();fig.savefig(FIGURES/'test_accuracy.png',bbox_inches='tight');plt.close(fig)
    example_ids=[];selection=[]
    def add(i,why):
        if int(i) not in example_ids:example_ids.append(int(i));selection.append(why)
    add(0,'First official test item, fixed ordinary case')
    if valid:
        lead=sorted(valid,key=lambda r:(-r['test']['accuracy'],r['model_key']))[0]
        with np.load(DATA/lead['candidate_folder']/'predictions.npz') as p:pred=p['test_pred'];scores=p['test_scores']
        gold=dataset['test'][:,-1]
        for mask,why in [(pred==gold,'First correct case for highest observed score'),(pred!=gold,'First error for highest observed score')]:
            indices=np.flatnonzero(mask)
            if len(indices):add(indices[0],why)
        sorted_scores=np.sort(scores,axis=1);add(np.argmin(sorted_scores[:,-1]-sorted_scores[:,-2]),'Smallest uncalibrated top-two score margin')
        if len(valid)>1:
            with np.load(DATA/valid[0]['candidate_folder']/'predictions.npz') as p:one=p['test_pred']
            with np.load(DATA/valid[1]['candidate_folder']/'predictions.npz') as p:two=p['test_pred']
            indices=np.flatnonzero(one!=two)
            if len(indices):add(indices[0],'First disagreement between first two readable model submissions')
    for i in range(len(dataset['test'])):
        if len(example_ids)>=5:break
        add(i,'First unused official test item to complete five distinct examples')
    examples=[]
    for i,why in zip(example_ids,selection):
        row={'id':f'test-{i:04d}','index':i,'gold':int(dataset['test'][i,-1]),'selection':why,'predictions':{}}
        for r in valid:
            with np.load(DATA/r['candidate_folder']/'predictions.npz') as p:row['predictions'][r['label']]=int(p['test_pred'][i])
        examples.append(row)
    save_json(DATA/'worked_examples.json',examples)
    fig,axes=plt.subplots(1,len(examples),figsize=(12,4))
    for ax,case in zip(np.atleast_1d(axes),examples):
        ax.imshow(dataset['test'][case['index'],:64].reshape(8,8),cmap='Greys',vmin=0,vmax=16);ax.axis('off')
        ax.set_title(f'{case["id"]}\nGold {case["gold"]}',fontsize=10)
        text='\n'.join(f'{name}: {pred}' for name,pred in case['predictions'].items()) or 'No scored submission yet'
        ax.text(.5,-.08,text,ha='center',va='top',transform=ax.transAxes,fontsize=9)
    fig.suptitle('Traceable test images expose successes, errors and disagreements');fig.tight_layout()
    fig.savefig(FIGURES/'worked_examples.png',bbox_inches='tight');plt.close(fig)
    if valid:
        lead=sorted(valid,key=lambda r:(-r['test']['accuracy'],r['model_key']))[0];cm=np.asarray(lead['test']['confusion'])
        fig,ax=plt.subplots(figsize=(7,6));im=ax.imshow(cm,cmap='Blues')
        for i in range(10):
            for j in range(10):
                if cm[i,j]:ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8,color='white' if cm[i,j]>cm.max()/2 else '#222')
        ax.set(xticks=range(10),yticks=range(10),xlabel='Predicted digit',ylabel='Gold digit',title=f'{lead["label"]}: which digit classes are confused')
        fig.colorbar(im,ax=ax,label='Number of official test images');fig.tight_layout();fig.savefig(FIGURES/'confusion.png',bbox_inches='tight');plt.close(fig)
    summary={'results':results,'nearest_neighbour':baseline,'legacy_replay':legacy,'readable_submissions':len(valid),
             'unreadable_submissions':sum(not r.get('pending') and r.get('format_failure') is not None for r in results),
             'pending_submissions':sum(bool(r.get('pending')) for r in results),'n_test':1797}
    save_json(DATA/'study_summary.json',summary)
    return summary



import math,re
COUNTS={"W1":4096,"b1":64,"W2":2048,"b2":32,"W3":320,"b3":10}

def indexed_entries(text):
    """Parse only unambiguous complete numeric members, even with an unfinished outer JSON.

    A numeric token at end of text is not accepted without its following comma/brace,
    because an output limit could have cut the number itself. No values are inferred.
    """
    text=text.strip();decoder=json.JSONDecoder();pos=0;found={k:{} for k in COUNTS};issue=None
    def whitespace(i):
        while i<len(text) and text[i].isspace():i+=1
        return i
    def take(i):return decoder.raw_decode(text,whitespace(i))
    try:
        pos=whitespace(pos)
        if pos>=len(text) or text[pos]!='{':return found,{'kind':'not_json_object','partial':False}
        pos+=1;seen_tensors=set()
        while True:
            pos=whitespace(pos)
            if pos>=len(text):return found,{'kind':'unfinished_outer_object','partial':True}
            if text[pos]=='}':
                pos=whitespace(pos+1)
                assert pos==len(text),'Text after the JSON object'
                return found,None
            name,pos=take(pos);assert isinstance(name,str) and name in COUNTS,'Unknown tensor name'
            assert name not in seen_tensors,'Duplicate tensor object';seen_tensors.add(name)
            pos=whitespace(pos);assert pos<len(text) and text[pos]==':','Expected tensor colon';pos=whitespace(pos+1)
            assert pos<len(text) and text[pos]=='{','Expected numbered tensor object';pos+=1
            while True:
                pos=whitespace(pos)
                if pos>=len(text):return found,{'kind':'unfinished_tensor','tensor':name,'partial':True}
                if text[pos]=='}':pos+=1;break
                index,pos=take(pos)
                assert isinstance(index,str) and re.fullmatch(r'[1-9][0-9]*',index),'Index must be a one-based integer string'
                number=int(index);assert 1<=number<=COUNTS[name],'Out-of-range index'
                assert number not in found[name],'Duplicate indexed entry within reply'
                pos=whitespace(pos);assert pos<len(text) and text[pos]==':','Expected value colon';value_start=whitespace(pos+1)
                value,end=take(value_start);after=whitespace(end)
                if after>=len(text):return found,{'kind':'numeric_token_without_delimiter','tensor':name,'index':number,'partial':True}
                if text[after] not in ',}':
                    tail=text[value_start:].strip()
                    if re.fullmatch(r'-?(?:0|[1-9][0-9]*)(?:\.[0-9]*)?(?:[eE][+-]?[0-9]*)?',tail):
                        return found,{'kind':'unfinished_numeric_literal','tensor':name,'index':number,'partial':True}
                    raise AssertionError('Unexpected character after numerical value')
                assert type(value) in [int,float] and math.isfinite(value),'Non-finite or non-numeric value'
                found[name][number]=value
                if text[after]==',':pos=after+1
                else:pos=after+1;break
            pos=whitespace(pos)
            if pos>=len(text):return found,{'kind':'unfinished_outer_object','partial':True}
            if text[pos]==',':pos+=1
            elif text[pos]=='}':
                pos=whitespace(pos+1);assert pos==len(text),'Text after JSON';return found,None
            else:raise AssertionError('Expected tensor separator')
    except json.JSONDecodeError as exc:
        return found,{'kind':'unfinished_or_malformed_token','partial':True,'position':exc.pos}
    except (AssertionError,ValueError,TypeError,OverflowError) as exc:
        return found,{'kind':'invalid_structure','partial':False,'message':str(exc)}

def missing_ranges(merged):
    result={}
    for name,count in COUNTS.items():
        missing=[i for i in range(1,count+1) if i not in merged[name]];ranges=[]
        for i in missing:
            if ranges and i==ranges[-1][1]+1:ranges[-1][1]=i
            else:ranges.append([i,i])
        if ranges:result[name]=ranges
    return result

def verify_indexed_artifacts():
    checks=[];base=DATA/'indexed'
    if not base.exists():return checks
    for folder in sorted(base.iterdir()):
        assembled=folder/'assembled_response.json'
        if not assembled.exists():continue
        merged={k:{} for k in COUNTS};turns=[]
        for path in sorted(folder.glob('turn-*/response.json')):
            record=json.loads(path.read_text(encoding='utf-8'));entries,issue=indexed_entries(record.get('answer',''))
            for name,values in entries.items():
                for index,value in values.items():
                    assert index not in merged[name] or merged[name][index]==value,'Conflicting teacher entries'
                    merged[name][index]=value
            turns.append({'path':str(path.relative_to(DATA)),'cache_key':record['cache_key'],'parse_issue':issue})
        assert not missing_ranges(merged),'Assembled submission has missing source indices'
        flat=np.array([merged[name][i] for name,count in COUNTS.items() for i in range(1,count+1)],dtype=float)
        saved=json.loads(assembled.read_text(encoding='utf-8'));payload=json.loads(saved['answer'])
        assert np.array_equal(flat,np.asarray(payload['weights'],dtype=float)),'Assembly changed a source value'
        checks.append({'model_key':folder.name,'values_verified':len(flat),'teacher_values_modified':False,'source_turns':turns})
    save_json(DATA/'indexed_provenance_checks.json',checks)
    return checks

def indexed_workflow_metadata():
    base=DATA/'indexed'
    if not base.exists():return
    for folder in base.iterdir():
        paths=sorted(folder.glob('turn-*/response.json'))
        if not paths:continue
        records=[json.loads(p.read_text(encoding='utf-8')) for p in paths]
        metadata={'started_utc':records[0]['started_utc'],'finished_utc':records[-1]['finished_utc'],
                  'wall_seconds':sum(r['wall_seconds'] for r in records),'usage':{'per_turn':[r.get('usage') for r in records]},
                  'cost_usd':sum(r.get('cost_usd') or 0 for r in records) if all(r.get('cost_usd') is not None for r in records) else None,
                  'turn_cache_keys':[r['cache_key'] for r in records],'outer_requests':len(records),
                  'provider_errors':[r['error'] for r in records if r.get('error')]}
        save_json(folder/'workflow_summary.json',metadata)
        p=folder/'results.json'
        if p.exists() and not (folder/'assembled_response.json').exists():
            result=json.loads(p.read_text(encoding='utf-8'))
            if len(metadata['provider_errors'])==len(records):
                result.update(format_failure=f'{len(records)} provider errors before numerical output',failure_category='provider_unavailable')
            result.update(wall_seconds=metadata['wall_seconds'],usage=metadata['usage'],cost_usd=metadata['cost_usd'])
            save_json(p,result)

def reported_billing():
    batches=[]
    for path in sorted(DATA.glob('**/batch_result.json')):
        if 'legacy' in path.parts:continue
        usage=json.loads(path.read_text(encoding='utf-8')).get('usage') or {}
        if usage.get('cost') is None:continue
        cost=float(usage['cost']);batches.append({'source':str(path.relative_to(DATA)),'cost_usd':cost})
        result_path=path.parent/'results.json'
        if result_path.exists():
            result=json.loads(result_path.read_text(encoding='utf-8'));result.update(cost_usd=cost,cost_source='Cached batch_result.json/usage/cost');save_json(result_path,result)
    indexed=DATA/'indexed/gemini'
    costs=[b['cost_usd'] for b in batches if b['source'].startswith(str(Path('indexed/gemini')))]
    if costs and (indexed/'results.json').exists():
        result=json.loads((indexed/'results.json').read_text(encoding='utf-8'));result.update(cost_usd=sum(costs),cost_source='Sum of indexed per-turn batch billing');save_json(indexed/'results.json',result)
    claude=[]
    for area in ['responses','recovery']:
        path=DATA/area/'opus/response.json'
        if path.exists():claude.append(json.loads(path.read_text(encoding='utf-8')).get('cost_usd') or 0)
    save_json(DATA/'billing_summary.json',{'google_api_cost_usd':sum(b['cost_usd'] for b in batches),'google_batches':batches,
        'claude_session_list_price_equivalent_usd':max(claude,default=0),'other_cli_charges_usd':None,
        'nemotron_route':'Free-only with zero-price cap and no paid fallback','note':'API charges and CLI list-price equivalents are different measures.'})

def _offline_network_guard(*args, **kwargs):
    raise RuntimeError("Network requests are disabled in notebook replay; use the explicit collection script for new experiments.")
requests.sessions.Session.request = _offline_network_guard


In [2]:
# Revalidate only cached answers, never collect new ones.
for area in ['responses', 'recovery', 'indexed']:
    if not (DATA / area).exists():
        continue
    for folder in (DATA / area).iterdir():
        model_key = folder.name.split('-')[0]
        if model_key not in MODELS:
            continue
        cached = folder / ('assembled_response.json' if area == 'indexed' else 'response.json')
        if cached.exists():
            record = json.loads(cached.read_text(encoding='utf-8'))
            validate_and_score(model_key, record, out_override=folder)
summary = make_figures()
print(json.dumps({k:v for k,v in summary.items() if k != 'results'}, indent=2))
print('Fresh teacher calls: 0')
reported_costs = []
claude_cumulative = []
for row in summary['results']:
    value = row.get('cost_usd')
    if value is not None:
        (claude_cumulative if row['model_key'] == 'opus' else reported_costs).append(value)
print('Reported USD/list-price equivalents, avoiding duplicate Claude cumulative counters:', sum(reported_costs) + max(claude_cumulative, default=0.0))
print('Other CLI dollar charges: not reported, not assumed zero.')


{
  "nearest_neighbour": {
    "hidden_train": {
      "n": 1775,
      "correct": 1749,
      "accuracy": 0.9853521126760564
    },
    "test": {
      "n": 1797,
      "correct": 1750,
      "accuracy": 0.9738452977184195
    }
  },
  "legacy_replay": [
    {
      "model": "astra",
      "parameters": 17,
      "classification_n": 50000,
      "correct": 50000,
      "accuracy": 1.0,
      "regression_n": 50401,
      "regression_mae": 0.0
    },
    {
      "model": "nemotron-no-reasoning",
      "parameters": 17,
      "classification_n": 50000,
      "correct": 24952,
      "accuracy": 0.49904,
      "regression_n": 50401,
      "regression_mae": 0.3347491920881353
    },
    {
      "model": "astra",
      "parameters": 388,
      "classification_n": 20000,
      "correct": 20000,
      "accuracy": 1.0
    },
    {
      "model": "nemotron-free",
      "parameters": 388,
      "classification_n": 20000,
      "correct": 19286,
      "accuracy": 0.9643
    }
  ],
  "readable_subm

## 1. Replication scope and data

The original blog distinguishes reading behaviour from weights, repairing a trained network, and writing a network from scratch. Our direct-writing tests target the last of these. Its tool-free game-writing experiment uses a 600→32→32→6 tanh network, which contains 20,486 weights and biases by calculation from the published shape. Our seventeen-, 388- and 6,570-value MLPs are approximately 1,205, 53 and 3 times smaller, respectively. The tasks and activations differ, so the comparison tests the mechanism at smaller sizes rather than reproducing the published game scores.

| Blog capability | What our programme tests | Replication status |
|---|---|---|
| Write weights from a task specification | Known-rule MLPs and sample-based digit MLPs, with complete numeric output and no teacher tools | Functional small-network examples reproduced on our tasks |
| Repair useful pretrained behaviour | SQL revisions start from an untrained tiny sequence scaffold; frozen word vectors supply representations | Pretrained-network repair claim not tested equivalently |
| Identify a hidden behaviour from weights | The tester inspects submitted matrices after observing outputs | No blind weight-reading experiment conducted |

The positive claim supported here is that language-model-authored numerical vectors can implement useful functions without a tester-run optimizer. We have no matched trained-MLP baseline and have not reproduced the article's reinforcement-learning comparisons. The nearest-neighbour digit reference below is a different classifier, included to show useful information in the sample.

The complete programme is outlined below. Preparatory controls, functional failures and incomplete submissions are labelled separately; none is silently converted into evidence for or against the whole blog. This notebook develops the MLP evidence. The linked SQL notebook develops the language-generation tests and retains the initial language-model controls.

| Experiment | Network size | Observed outcome |
|---|---|---|
| Pretrained reference | 19,969 stored | MacroStories inspected; no preserved scored generation |
| Random language control | 10,000 editable | Ten untrained tokens; near-uniform scores |
| Sol SQL writing and revision | 2,208 editable + 55,500 frozen | Constant query: 1/8 exposed match; revision 0/12 |
| Astra SQL writing | 2,208 editable + 55,500 frozen | Completed run 0/16; earlier cancellation unscored |
| Free-model SQL writing | 2,208 editable + 55,500 frozen | Ultra malformed; Super 6 nonzeros, 0/16 |
| SQL decoding controls | 2,208 editable, unchanged | Zero schema matches in 1,845 outputs, 11 conditions |
| Bounded SQL weights | 2,208 editable + 55,500 frozen | All numerical checks pass; 0/16 SQL matches |
| Weight inspection | 2,208 SQL; 388 MLP; Pythia-14m reference | SQL score collapse; interpretable ring-network structure |
| Known-rule tiny MLPs | 17 per task | Astra 50,000/50,000 classification; regression error zero |
| Four-ring MLP | 388 | Astra 20,000/20,000; Nemotron 96.43%; boundary errors remain |
| Sample-based digit MLP | 6,570 | Astra 91.54%, Sol 79.13%, Muse 33.72%; other first replies unusable |
| Indexed output and continuation | 6,570 | Gemini: 6570 explicit values across two indexed replies; 1091/1797 test correct (60.71%). Nemotron: three upstream timeouts, no values. |

The [full experiment ledger](data/replication_ledger.json) records scope and outcome categories, including interruptions and provider failures.

The source is [Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), by E. Alpaydin and C. Kaynak, donated to UCI on June 30, 1998. UCI publishes 5,620 cases with sixty-four integer features. The data are licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The downloaded archive, individual text files and their SHA-256 hashes are saved in `data/source/` and the manifest. This notebook does not substitute scikit-learn's smaller packaged digits dataset.

Each image is represented by an eight-by-eight grid. Read the first eight numbers as its top row, the next eight as the second row, and so on. An ink count of zero represents a blank block and sixteen is the upper end of the count range. The final CSV entry is the class label. It is removed before inference. There are no names, natural-language descriptions or class labels hidden among the sixty-four model inputs.

The official files provide 3,823 training cases and 1,797 test cases. We leave the official test file intact. With the fixed seed 20261103, the visible subset contains 205 training images for each of digits zero through seven, and 204 each for eight and nine. These class counts sum to 2,048. The remaining 1,775 training cases provide a secondary hidden split. No label from either hidden split appears in the user packet.

Keeping the official test split avoids choosing favourable cases after observing weights. The source notes say thirty writers supplied training images and thirteen different writers supplied test images. Unused training images provide a separate check; combining these scores could hide a source discrepancy.

![One actual visible image per digit, with stable official training-row IDs](figures/visible_digits.png)

The ten panels show actual visible cases, one per label. Greyscale intensity represents the supplied ink counts, and the titles identify the original zero-based training row. A broad loop, a thin vertical stroke and a diagonal stroke leave different patterns, but the image size is small enough that some strokes merge. These examples make the input concrete without giving the teachers an additional image channel: every teacher receives the pixel values as text.

All six first requests share the same approximately 305 KB packet and prompt hash. No teacher receives a trained network, hidden labels, nearest-neighbour predictions or a test score.

No official test pixel row exactly duplicates a visible image. This excludes literal row copying, while similar handwriting, writer effects and pretraining exposure remain possible.

The owner's public-data exception is recorded before the teacher responses. The test measures whether the submitted weights recognize these digits; prior exposure remains possible.

## 2. The fixed network and numerical contract

The network has sixty-four inputs, hidden layers of sixty-four and thirty-two neurons, and ten output scores. A fully connected layer multiplies its inputs by a matrix and adds one bias per output neuron. The hidden activation is ReLU, which replaces negative intermediate values by zero. The final layer is linear, so its ten outputs are scores rather than calibrated probabilities.

| Tensor | Shape | Values | Zero-based offsets in the response |
|---|---|---:|---|
| W1 | 64 × 64 | 4,096 | 0–4,095 |
| b1 | 64 | 64 | 4,096–4,159 |
| W2 | 32 × 64 | 2,048 | 4,160–6,207 |
| b2 | 32 | 32 | 6,208–6,239 |
| W3 | 10 × 32 | 320 | 6,240–6,559 |
| b3 | 10 | 10 | 6,560–6,569 |

The six tensors contain 6,570 independent values. Matrix rows correspond to output neurons and columns to input neurons. Each is flattened row by row, then the tensors are concatenated in the table's order. Supplying a complete flat vector avoids disagreements about row nesting, but it makes counting and indexing part of the output task. A correct numerical idea with missing entries is still not an executable submission.

The owner later requested one-based numbered objects per tensor and continuation prompts for missing entries. This workflow preserves complete tensor/index/number members, checks all required indices, and asks only for missing ranges. Gemini's two replies cover all 6,570 entries with no conflicts. Every assembled value is verified against those original indexed members. Nemotron's three requests time out upstream before any numeric entry. The original flat-vector failures remain separate outcomes.

```python
import numpy as np
def preprocess(raw_pixels):
    # Exactly 64 integers, row-major flattening of the 8 by 8 image.
    # 0 is blank, 16 is the largest ink count. No fitted transform.
    return np.asarray(raw_pixels, dtype=np.float64) / 16.0
def network(raw_pixels, W1, b1, W2, b2, W3, b3):
    x = preprocess(raw_pixels)
    h1 = np.maximum(x @ W1.T + b1, 0.0)
    h2 = np.maximum(h1 @ W2.T + b2, 0.0)
    return h2 @ W3.T + b3
def postprocess(scores):
    # Digit labels are 0 through 9. Ties choose the lowest index.
    return np.argmax(scores, axis=-1)
```

This is the exact code in the teacher packet. Dividing by sixteen is a fixed scale conversion; it uses no estimated mean, standard deviation or hidden labels. The two hidden matrix products and ReLU operations are the only learned computations. The postprocessor selects a digit from the ten scores. It adds no handwriting-specific rules and never repairs a mistaken class.

For an explicitly illustrative score vector, suppose digit five had the largest score and digit eight the second largest. The prediction would be five, regardless of whether either score was positive. The difference between those two scores is a margin. This notebook can use a small margin to select a difficult example, but cannot interpret a margin of 0.9 as a 90% probability. No probability calibration data are supplied.

Each teacher is asked to return only `{"weights":[...]}` with exactly 6,570 explicit finite JSON numbers. Booleans, missing entries, extra object keys, symbolic expressions and non-finite constants are rejected. The program never pads a short vector, cuts a long one to size, replaces suspicious values or infers a repeated block. Such repairs would turn the tester into a second parameter author and make success difficult to attribute.

The first-call failures prompted documented transport recoveries. Google rejected the exact-length JSON grammar before generating a token, so its first recovery keeps the user prompt unchanged and simplifies only the provider schema. That high-reasoning request then exhausted a 32,768-token emitted response without final weights. A final low-reasoning variant returned 6,013 numbers instead of 6,570. Its vector is retained but cannot be loaded into the specified architecture. Claude first exhausted its output/budget allowance, then its saved-session continuation returned a refusal citing a provider safety classifier. No Claude weights were available, and no further Claude retries were attempted. These are service/completion outcomes rather than observed digit-classification errors.

After parsing, the vector is reshaped in float64 and saved in `candidate.npz`. Every saved array is compared exactly with the corresponding response slice. The frozen forward code then produces scores for visible and hidden images. Non-finite forward scores are flagged as an unusable network rather than converted into predictions. A candidate can pass the JSON contract and still be a poor classifier; those outcomes are reported separately.

Unlike the bounded SQL trial, this MLP experiment has no density rule or requirement that weights resemble a Gaussian distribution. Sparse networks and zero biases can be sensible numerical programs. The prompt asks for moderate coefficients but permits every finite value. We inspect their distribution after generation rather than changing their values to match an initializer. The output numbers alone determine whether a useful function is implemented.

## 3. Model access and isolation

| Teacher | Pinned version or runtime ID | Provider and call | Published size/precision | Output settings | Why included |
|---|---|---|---|---|---|
| GPT-6 Astra | gpt-6-astra; revision not exposed | OpenAI, codex exec | Not disclosed | xhigh; legacy 17-value medium; no tools | Requested author; known-rule successes |
| GPT-6.1 Sol | gpt-6.1-sol; immutable revision not exposed | OpenAI, codex exec | Not disclosed | xhigh; tools disabled | Requested second OpenAI numerical author |
| Claude Opus 5.5 | claude-opus-5-5; immutable revision not exposed | Anthropic, claude -p | Not disclosed | xhigh first; high continuation; tools disabled | Requested independent vendor; recovery kept separate |
| Muse Wonder Full | muse-wonder-full-1.0; immutable revision not exposed | Meta, muse exec | Not disclosed | xhigh; empty preset; tools disabled | Requested independent vendor |
| Nemotron 3.5 Super VL | nvidia/nemotron-3.5-super-vl-20261015 | NVIDIA, OpenRouter free | 120B total, 12B active; precision not attested | Reasoning first; off indexed; 131,072 cap; free only | Requested open-weight author; 388-value comparison |
| Gemini 4 Argon batch | google/gemini-4-argon-20261013 | Google Vertex, OpenRouter Batch API | Not disclosed | high original and schema retry; low final variant; 131,072 requested output cap | Requested Google comparison; asynchronous batch route |

The CLI model names are the exact requested identifiers, but immutable dated weight revisions are not exposed for them. Where the transport does not attest a snapshot, the table says so instead of presenting an alias as a pinned checkpoint. OpenRouter metadata pins canonical dated slugs before requests are sent. Published parameter count and numerical precision of the teacher models are recorded only where available; the candidate MLP always runs in float64.

```mermaid
flowchart TD
    P["One frozen packet<br/>2,048 labelled rows + code"] --> A["Astra: codex exec, xhigh<br/>weights only"]
    P --> S["Sol: codex exec, xhigh<br/>weights only"]
    P --> C["Opus 5.5: claude -p, xhigh<br/>weights only"]
    P --> W["Muse Wonder Full: muse exec, xhigh<br/>weights only"]
    P --> N["Nemotron: OpenRouter free<br/>weights + emitted reasoning fields"]
    P --> G["Gemini Argon: OpenRouter Batch API, high<br/>asynchronous result"]
    C -->|"Output/budget stop"| CC["Opus: claude -p, high continuation<br/>saved context; refusal"]
    G -->|"Schema rejected before decoding"| GR["Gemini: Batch API, high<br/>simpler transport schema"]
    GR -->|"Thinking/output limit"| GL["Gemini: Batch API, low<br/>same user packet"]
    N -->|"Output limit"| NI["Nemotron: OpenRouter, reasoning off<br/>numbered tensor entries"]
    GL -->|"Wrong count"| GI["Gemini: Batch API, low<br/>numbered tensor entries"]
    NI -->|"Missing indices; CONTINUE, at most 3 calls"| NI
    GI -->|"Missing indices; CONTINUE, at most 3 calls"| GI
    A --> V["Independent tester waits for each response<br/>no cross-model messages or feedback"]
    S --> V
    C --> V
    W --> V
    N --> V
    G --> V
    CC --> V
    GR --> V
    GL --> V
    NI --> V
    GI --> V
```

The six calls are independent and can run concurrently. Concurrency does not give teachers a shared workspace or access to one another's replies. The CLI working directories are empty runtime directories, not notebook folders. Teacher processes are configured without shell, code execution, file tools, web search, skills or callable agents. Their input comes from the supplied prompt or a prompt file read by the CLI transport itself.

This distinction between the transport and the model is essential. The tester must read a file to send a large prompt and must write the answer somewhere. That does not give the model a file-reading or file-writing tool. The model's task is to emit text containing numbers. The parent harness alone stores those numbers in the required format and evaluates them.

Startup records and events audit isolation: Codex uses neutral isolated instructions, Claude reports empty tool and skills catalogues, and Muse uses an empty preset. Mandatory built-in metadata and exact switches remain in the protocol and raw logs. Actual tool-use events invalidate a submission.

The user packets are identical, but mandatory provider system scaffolding and output enforcement are not identical. Codex, Muse and OpenRouter can enforce a JSON shape through their native options; Claude is asked for the same JSON contract in the prompt. Reasoning-effort labels also differ by provider. These differences are part of this practical transport comparison and prevent a claim that only teacher weights or reasoning effort varied.

## 4. How the study got here

The earlier tiny MLP had two inputs, four ReLU hidden neurons and one output, for seventeen parameters. It was given a complete rule: classify opposite signs as class one and matching signs as class zero, or compute `abs(x1) + 0.5*abs(x2)`. Zero coordinates were excluded from classification. Astra's emitted numbers achieved 50,000/50,000 correct classifications and zero absolute error on 50,401 regression points. Those are measured results on those rules, not a promise that a larger data-only task will be easy.

One saved measured example is `(-0.8, 0.3)`, the first example in the archived seventeen-parameter results. Opposite signs require class one. Astra's score is 0.6000000000000001, above the fixed zero threshold, so its class is one. The regression target is `0.8 + 0.5*0.3`; both the cached target and its output are 0.9500000000000001 in float64. This is one exact success, with the aggregate checks providing coverage beyond the example.

The next network had two hidden layers of sixteen neurons and four outputs, for 388 parameters. Its complete rule assigned points in the unit disk to four rings using radius boundaries 0.25, 0.50 and 0.75. Astra classified all 20,000 balanced random points correctly; free Nemotron classified 19,286/20,000 correctly, or 96.43%. The dense-grid Astra score was 125,601/125,625. The twenty-four grid errors were concentrated at exact boundaries, so the random-sample ceiling did not establish an exact circle implementation.

The working Astra ring network can be interpreted as an approximation to radius followed by thresholds. Its many zero values and low-rank blocks therefore have a functional explanation. This is why the digits experiment does not ban sparse constructions. In a deliberately difficult narrow boundary stress set, its score was 34,784/43,200, or 80.52%; that stress score samples special slivers around thresholds and should not be substituted for ordinary disk accuracy.

![Cache-only reproduction of the earlier known-rule classification trials](figures/legacy_accuracy.png)

The two panels show different tasks and are not a size-controlled comparison. The seventeen-value task has two labels and 50,000 classification points; its dashed chance reference is 50%. The ring task has four labels and 20,000 balanced points, with a 25% chance reference. Astra reaches the measured ceiling in both samples. Nemotron's earlier reasoning-off tiny trial is much weaker than its later reasoning-enabled ring trial, but task and settings changed together. The notebook recomputes these counts from the archived arrays without asking either teacher for a new response.

All historical prompts, responses, numeric arrays, point sets and results are archived under `data/legacy/`, with a file-level migration manifest. These earlier trials informed the harder question but are not six-model comparison rows. They differ in teacher set, effort, task information and sample construction. The larger experiment adds sixty-four visual features, ten labels, seventeen times as many parameters and a sample in place of a complete label formula.

## 5. Results and comparisons

Before comparing accuracies, check whether each response could be used as a network. A blank reply, unfinished stream or malformed vector is not a set of digit predictions. The table separates those failures from readable but wrong predictions. It also keeps requests still running separate from failures. A pending row cannot be interpreted as zero accuracy.

| Submission | Workflow | Official test | Wilson 95% | Visible accuracy | Unused training | Format/service failures |
|---|---|---|---|---|---|---|
| Astra | First submission | 1645/1797 (91.54%), 152 wrong | 90.16–92.74% | 93.16% | 94.03% | 0 |
| Sol | First submission | 1422/1797 (79.13%), 375 wrong | 77.19–80.95% | 82.91% | 83.61% | 0 |
| Opus 5.5 | First submission | Thinking/output limit, then budget exhaustion | N/A | N/A | N/A | 1 unusable response |
| Muse Wonder | First submission | 606/1797 (33.72%), 1191 wrong | 31.57–35.94% | 37.74% | 37.86% | 0 |
| Nemotron | First submission | Output limit; reasoning only; no weights | N/A | N/A | N/A | 1 unusable response |
| Gemini batch | First submission | Provider rejected schema before decoding | N/A | N/A | N/A | 1 unusable response |
| Opus 5.5 continuation | Recovery | Refusal citing provider safety classifier; no weights | N/A | N/A | N/A | 1 unusable response |
| Gemini batch schema retry | Recovery | Incomplete or failed model response | N/A | N/A | N/A | 1 unusable response |
| Gemini batch low effort | Recovery | Expected 6570 weights; received 6013 | N/A | N/A | N/A | 1 unusable response |
| Gemini batch indexed | Indexed, 2 calls | 1091/1797 (60.71%), 706 wrong | 58.43–62.95% | 61.82% | 63.83% | 0 |
| Nemotron indexed | Indexed, 3 calls | 3 provider errors before numerical output | N/A | N/A | N/A | 3 provider errors |

![Official test accuracy, with pending or unreadable submissions explicitly labelled](figures/test_accuracy.png)

Blue bars show accuracy on the 1,797 official test images. The brown reference bar shows one-nearest-neighbour using only the same visible images. The dashed line at 10% is uniform random guessing over ten labels, not a fitted baseline. Error bars are Wilson 95% intervals for each fixed network's image-level correctness. They do not include uncertainty from making another teacher request, selecting another visible sample or choosing different writers.

The highest first-attempt score observed is Astra at 1645/1,797 (91.54%). The nearest-neighbour reference correctly labels 105 more images. Astra and Sol differ by 223 correctly labelled images on this fixed test set. These are differences between single submitted networks. They do not establish a stable ranking across repeated requests or new datasets. The separately labelled Gemini batch indexed scored 1091/1,797 (60.71%) after its documented transport recovery.

The visible-sample score is useful because a network can fail even to represent the examples it was given. A high visible score with a substantially lower hidden score would instead suggest sample-specific behaviour or a shift between sources. Neither pattern proves its cause without more evidence. The unused-training score helps distinguish a discrepancy confined to the official test split from a broader failure on unseen examples.

One-nearest-neighbour achieves 1,749/1,775, or 98.54%, on unused training images and 1,750/1,797, or 97.38%, on official test images. It stores all 2,048 visible images and compares them at prediction time. The generated MLP gets no such lookup operation and has only 6,570 numerical parameters. Therefore the reference shows that the sample contains useful local structure, but does not establish that this architecture and a tool-free numerical author should match it.

With only one attempt per model, differences describe the submitted artefacts, not stable capacities of the model families. A response that happens to be well structured can dominate a single comparison even if another generation would differ. If several methods sit near a floor or ceiling, the test should be treated as unable to rank them there. The raw count of separating images is more informative than ordering nearly identical percentages.

## 6. Worked test cases and error analysis

Examples below are drawn from fixed official test row IDs. The first case is always the first test row. Once readable submissions exist, selection adds the first success and first error for the highest observed score, the smallest uncalibrated top-two margin, and a disagreement between the first two readable submissions when available. The procedure fills any missing categories with the first unused cases. These rules are explicit so striking examples do not masquerade as a representative sample.

![Five actual official test images with gold labels and predictions from available networks](figures/worked_examples.png)

### Case 1: `test-0000`, gold digit 0

Selection: First official test item, fixed ordinary case. Observed outputs: Astra → 0, Sol → 0, Muse Wonder → 8, Gemini batch indexed → 0. The shown predictions disagree with the gold label for at least one network. This item contributes one count per scored network; the panel above shows its actual pixel grid. The stable row index connects it to `data/source/optdigits.tes` and the saved prediction arrays.

### Case 2: `test-0002`, gold digit 2

Selection: First error for highest observed score. Observed outputs: Astra → 8, Sol → 1, Muse Wonder → 1, Gemini batch indexed → 1. The shown predictions disagree with the gold label for at least one network. This item contributes one count per scored network; the panel above shows its actual pixel grid. The stable row index connects it to `data/source/optdigits.tes` and the saved prediction arrays.

### Case 3: `test-0415`, gold digit 9

Selection: Smallest uncalibrated top-two score margin. Observed outputs: Astra → 3, Sol → 9, Muse Wonder → 3, Gemini batch indexed → 9. The shown predictions disagree with the gold label for at least one network. This item contributes one count per scored network; the panel above shows its actual pixel grid. The stable row index connects it to `data/source/optdigits.tes` and the saved prediction arrays.

### Case 4: `test-0001`, gold digit 1

Selection: First unused official test item to complete five distinct examples. Observed outputs: Astra → 1, Sol → 1, Muse Wonder → 1, Gemini batch indexed → 1. Every available prediction shown matches the gold label. This item contributes one count per scored network; the panel above shows its actual pixel grid. The stable row index connects it to `data/source/optdigits.tes` and the saved prediction arrays.

### Case 5: `test-0003`, gold digit 3

Selection: First unused official test item to complete five distinct examples. Observed outputs: Astra → 3, Sol → 8, Muse Wonder → 1, Gemini batch indexed → 3. The shown predictions disagree with the gold label for at least one network. This item contributes one count per scored network; the panel above shows its actual pixel grid. The stable row index connects it to `data/source/optdigits.tes` and the saved prediction arrays.

### End-to-end calculation for `test-0000`

The first eight raw ink counts are `[0, 0, 5, 13, 9, 1, 0, 0]`. Dividing by sixteen gives `[0.0, 0.0, 0.3125, 0.8125, 0.5625, 0.0625, 0.0, 0.0]` for the first input row. The complete input remains sixty-four numbers. Astra's unchanged matrices yield the following ten output scores.

| Digit | Observed score |
|---|---|
| 0 | 86.5 |
| 1 | 67.0625 |
| 2 | 72.875 |
| 3 | 73.5625 |
| 4 | 73.5 |
| 5 | 76.0625 |
| 6 | 74.25 |
| 7 | 67.6875 |
| 8 | 75.0625 |
| 9 | 78.6875 |

The largest score has digit 0; the hidden label is 0. This contributes 1 to that network's correct count. No score or label is fed back into its parameters.

The panels show the actual sixty-four-value inputs before division by sixteen. Success means the largest score has the gold label. Any distinction between similar loops or strokes must arise from the supplied matrices and biases; the displayed gold labels were absent from the hidden packet.

![Confusion matrix for highest observed score](figures/confusion.png)

Rows are gold digits and columns are predictions. Each cell counts official test images, and the diagonal counts correct answers. For Astra, 13 of 180 true digit-9 images were labelled 5; 12 of 180 true digit-9 images were labelled 3. These are the two largest off-diagonal counts in this submitted network. The figure is selected descriptively by the highest observed score, not as an independently validated winner. Counts identify recurring class confusions; they do not establish why the errors occur.

Selection is descriptive: a small margin can accompany a correct or incorrect answer. No human relabelling is performed, and unobserved failure categories are never filled with invented cases.

An end-to-end case starts with a raw sixty-four-entry row. Its entries are scaled, passed through the two hidden layers and converted to ten scores by the final affine layer. Argmax yields one integer. Comparing that integer to the hidden label contributes exactly one correct or incorrect count to the accuracy numerator. It does not affect any weight, later prediction or request. Prediction arrays retain all ten scores so an investigator can inspect this sequence without asking the teacher to run again.

## 7. Reasoning logs and parameter patterns

| Submission | Emitted reasoning text | Reasoning tokens | Raw evidence |
|---|---|---|---|
| Astra | No readable reasoning text emitted | 16471 | [Raw evidence](data/responses/astra/response.json) |
| Sol | No readable reasoning text emitted | 21633 | [Raw evidence](data/responses/sol/response.json) |
| Opus 5.5 | No readable reasoning text emitted | 64000 | [Raw evidence](data/responses/opus/response.json) |
| Muse Wonder | No readable reasoning text emitted | 16786 | [Raw evidence](data/responses/muse/response.json) |
| Nemotron | [160,543 characters](data/responses/nemotron/reasoning.txt) | 131072 | [Raw evidence](data/responses/nemotron/response.json) |
| Gemini batch | No readable reasoning text emitted | Not reported | [Raw evidence](data/responses/gemini/response.json) |
| Opus 5.5 continuation | No readable reasoning text emitted | 1007 | [Raw evidence](data/recovery/opus/response.json) |
| Gemini batch schema retry | No readable reasoning text emitted | 32765 | [Raw evidence](data/recovery/gemini/response.json) |
| Gemini batch low effort | No readable reasoning text emitted | 0 | [Raw evidence](data/recovery/gemini-low/response.json) |
| Gemini batch indexed | No readable reasoning text emitted | 0 | [Raw evidence](data/indexed/gemini/assembled_response.json) |
| Nemotron indexed | No readable reasoning text emitted | Not reported | [Raw evidence](data/indexed/nemotron/workflow_summary.json) |

The notebook preserves provider-emitted reasoning because it can show the numerical construction a teacher describes: an intended feature detector, an indexing plan, a simplification or a decision to use sparse blocks. These logs are observations of emitted text. They are not a guarantee that every intermediate computation was exposed, and a plausible explanation does not validate a parameter vector. The independent forward pass remains the test of what the final numbers do.

Claude's `thinking_delta` events and final thinking blocks are retained when present. Nemotron's reasoning text and reasoning-detail fields are retained as streamed. Muse's raw events preserve any reasoning fields its CLI chooses to expose. Codex may provide a reasoning-token count without the full text. The analysis does not try to reconstruct absent content or substitute a narrative written by the tester.

The raw events are the source of truth; `reasoning.txt` is a convenience rendering. Opaque signatures stay opaque. Timing and usage accompany every completed response, without assuming that longer reasoning implies a better network.

Weight summaries count exact zeros and describe magnitudes after generation. A large fraction of zeros can mean unused neurons, a deliberately sparse program or an unhelpful constant predictor. A dense matrix can be just as useless. The interpretation must connect a pattern to the forward computation or observed predictions. No weights are rescaled merely to make a plot look like one from a conventionally trained model.

| Submission | Exact zeros / 6,570 | Nonzero first-layer rows / 64 | Nonzero second-layer rows / 32 | Largest magnitude |
|---|---|---|---|---|
| Astra | 5344 | 40 | 20 | 40.625 |
| Sol | 5546 | 40 | 20 | 19.25 |
| Muse Wonder | 6257 | 10 | 10 | 12.75 |
| Gemini batch indexed | 5438 | 64 | 20 | 2.7 |

![Nonzero coefficients by layer for every scored submitted network](figures/parameter_use.png)

The stacked bars count nonzero coefficients in each affine layer, including its bias. The total contract remains 6,570 values for every network; the bars show how many are not exactly zero. A nonzero row is a structural count, not proof that the neuron activates on an image. It also differs from a mathematical count of minimal functional degrees of freedom. These diagnostics explain utilisation without equating more nonzero entries with a better classifier.

For the first Astra and Sol submissions, forty of the sixty-four first-layer rows and twenty of the thirty-two second-layer rows are nonzero. Muse uses ten rows in each hidden layer. Their official test accuracies differ considerably, but this observation cannot establish that using fewer neurons caused a lower score. The actual numerical detectors also differ. The values and structural counts are cached in `data/parameter_use.json` for direct inspection.

Astra and Sol have exactly identical second-layer and output matrices and biases. Their different predictions therefore come from the first matrix and its biases. Inspection of those common downstream arrays gives a more specific description: for each digit, group four first-layer ReLU outputs as `a,b,c,d`; the final digit score is `max(a+b, c+d)`. The second layer computes `ReLU(a+b-c-d)` and `c+d`, and the final layer adds them. Since all four first-layer outputs are nonnegative, that sum equals the stated maximum. This is an interpretation of observable weights, not a reconstruction of hidden reasoning or an alternative inference implementation.

The notebook checks the equivalent expression against the ordinary full network scores on all 1,797 test images and saves the largest numerical differences in `data/program_structure.json`. The interpretation explains why dozens of hidden slots can remain unused while a nontrivial classifier still runs. It also narrows the comparison between these two particular submissions to different first-layer numerical features. It does not establish that the models computed class means, used a particular learning algorithm internally or would choose the same construction on another request.

Structural interpretation never repairs a transposition, fills a row or changes a coefficient. The emitted numbers remain the object being evaluated.

## 8. Limits and practical meaning

The strongest data limitation is the benchmark's age. These public files were donated in 1998, long before the current teachers were trained. A model may have seen the data, examples, class prototypes or code for the benchmark. Hiding rows from the prompt does not make them historically secret. A fresh digit collection or a carefully transformed new task is needed before claiming learning from an unseen dataset.

The strongest design limitation is one generated network per model. Image-level intervals quantify finite test cases while holding that network fixed. They do not quantify variation between independent numerical submissions. A convincing capacity comparison would repeat requests and sample selection, preserve every generated vector, and summarize variability across networks rather than treating thousands of image predictions as independent teacher trials.

The visible sample is almost balanced by digit; the official test split has its own class counts. Accuracy therefore weights test labels by their observed frequencies. Per-class results help identify a model that mainly predicts one or two digits. They also prevent an aggregate from hiding a class with little support. This remains a single dataset, so the score does not transfer automatically to higher-resolution images, colour images or real document OCR.

The architectural choice is an experimental constraint. Sixty-four hidden neurons followed by thirty-two permit many piecewise linear computations, but no optimizer is run to establish the best achievable accuracy for this exact architecture. The nearest-neighbour reference establishes useful information in the sample, not an MLP ceiling. This study evaluates direct weight writing under the chosen contract, not the theoretical capacity of all networks of this size.

Output length is part of the challenge. Writing 6,570 explicit numbers can require tens of thousands of tokens before reasoning. A failed format may reflect counting, indexing or response-length limits rather than an inability to describe a good algorithm. The caches separate these outcomes, but the one-shot design cannot determine how much a retry, a block-wise interface or a more permissive representation would help.

These results concern numerical writing with teacher tools disabled. Allowing execution, fitting or intermediate evaluation would be a different workflow, including settings explored by the original blog.

## 9. Reproduction, caches and spend

The executed [notebook](41_direct_weight_mlps.ipynb) defines its helpers internally, blocks network requests, and replays local caches to reproduce predictions, figures and reported spend. Initial requests are in `data/responses/`, earlier recoveries in `data/recovery/`, and numbered chunks in `data/indexed/`. Source images, sample membership and hidden labels are committed for audit.

Install Python with NumPy, SciPy, matplotlib, requests, nbformat, nbconvert and an IPython kernel. From this folder, run `jupyter nbconvert --to notebook --execute --inplace 41_direct_weight_mlps.ipynb`. Replay requires no model credential. The complete hidden files are available to the reader; the isolation and prompt hash are what establish which rows each teacher received during generation.

Fresh collection is separate: `python experiment.py collect --models astra --max-new-calls 1`. Started markers prevent accidental duplicate requests. `indexed_recovery.py` implements the owner-requested indexed continuation workflow. Inspect existing logs before resuming; credentials stay in local authentication or the environment.

| Submission | Native input tokens | Native output tokens | Wall time | Reported USD or list-price equivalent |
|---|---|---|---|---|
| Astra | 270858 | 30414 | 546.2 s | Not reported |
| Sol | 270858 | 35562 | 529.9 s | Not reported |
| Opus 5.5 | 2 | 64000 | 511.0 s | $5.602564 (cumulative session list-price equivalent) |
| Muse Wonder | 282926 | 30415 | 424.3 s | Not reported |
| Nemotron | 302686 | 131072 | 2113.7 s | $0.000000 |
| Gemini batch | Not reported | Not reported | 371.1 s | $0.000000 |
| Opus 5.5 continuation | 4 | 1158 | 16.6 s | $5.741595 (cumulative session list-price equivalent) |
| Gemini batch schema retry | 302875 | 32768 | 709.8 s | $0.466715 |
| Gemini batch low effort | 302875 | 13723 | 610.1 s | $0.371490 |
| Gemini batch indexed | 662718 | 45237 | 1386.4 s | $0.888903 |
| Nemotron indexed | Not reported | Not reported | 364.4 s | Not reported |

Unknown CLI charges are not treated as zero. Claude's dollar counters are cumulative: its final session total is US$5.741595, including the earlier US$5.602564 stop. Its budget flag allowed an overshoot. Nemotron is restricted to a zero-price route without paid fallback. Google batch IDs permit polling without resubmission. Raw usage retains provider-specific token counters.

The [protocol](data/PROTOCOL.md) records design changes, recovery settings and parser rules. The [manifest](data/manifest.json) pins inputs and model identities; the archive manifest hashes historical files. Replay sends no traces; fresh calls use optional local Langfuse when available.

## 10. Future work and further reading

Links below were checked on November 3, 2026.

### Future work

- Reproduce a published game-writing task with matched activations, evaluation and a trained baseline, then reduce network size. Measure which published performance comparisons survive; the current positive results use different tasks.
- Add a blind weight-reading test using matched networks for different known rules. Ask teachers to identify behaviour without outputs, rather than relying on the tester's post-hoc interpretation.
- Start with a competent pretrained small network and a defined defect, then request targeted edits. Measure retained skill and repaired behaviour; our SQL scaffold lacks the trained sequence core used in the blog's repair setting.
- Compare flat and indexed outputs across repeated requests with matched reasoning and total output budgets. Measure format completion and conditional accuracy; our recovery workflows change several transport settings together.
- Use newly collected handwriting or a preregistered new labelling and multiple visible samples. Measure variation between submitted networks and reduce the public-data exposure caveat.

### Further reading

- [Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) (Alpaydin and Kaynak, 1998). The source dataset and its official train/test files.
- [Sparks of Weight-Space Programming](https://rikiyat.github.io/blog/weight-space-programming/) (Takehi et al., 2026). The motivating distinction between directly writing numerical networks and editing existing pretrained parameters.
- [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html) (Glorot and Bengio, 2010). Explains scale considerations for network initialization, without making a histogram a test of functional correctness.
- [Git Re-Basin: Merging Models modulo Permutation Symmetries](https://arxiv.org/abs/2209.04836) (Ainsworth et al., 2023). Shows why neuron coordinates and raw parameter patterns need functional interpretation.
- [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html) (Guo et al., 2017). Explains why classifier scores and margins should not be read as reliable confidence probabilities.